In [1]:
# Cell 1. Pre-sleep Design C Stage 1 training setup and tensor validation
# 원하는 결과:
# - Design C Stage 1 final MLP tensor를 로드한다.
# - train/validation/test shape, target distribution, NaN/Inf를 확인한다.
# - 학습 실행 flag는 기본 False로 둔다.
# - 아직 학습은 하지 않는다.

from pathlib import Path
import json
import random
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

STAGE1_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage1"
TENSOR_DIR = STAGE1_DIR / "mlp_current_day_final"
METADATA_PATH = STAGE1_DIR / "metadata.json"
FEATURE_COLUMNS_PATH = STAGE1_DIR / "pre_sleep_design_c_stage1_final_feature_columns.csv"

OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_mlp_outputs"
MODEL_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_PRE_SLEEP_STAGE1_TRAINING = False

SEED = 2026
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EXPERIMENT_ID = "presleep_stage1_000"
MODEL_FAMILY = "mlp_current_day"
FEATURE_TIMING = "pre_sleep"
SUBSET_NAME = "design_c_stage1_intraday_previous_day"
WINDOW = 1

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

def load_npz_split(split_name):
    path = TENSOR_DIR / f"{split_name}.npz"
    data = np.load(path, allow_pickle=True)
    return {
        "path": path,
        "X": data["X"].astype(np.float32),
        "y": data["y"].astype(np.int64),
        "sleep_episode_id": data["sleep_episode_id"].astype(str),
        "participant_object_id": data["participant_object_id"].astype(str),
        "feature_columns": data["feature_columns"].astype(str),
    }

split_data = {
    split_name: load_npz_split(split_name)
    for split_name in ["train", "validation", "test"]
}

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
feature_columns_df = pd.read_csv(FEATURE_COLUMNS_PATH, encoding="utf-8-sig")

loaded_feature_columns = split_data["train"]["feature_columns"].tolist()

tensor_summary_rows = []

for split_name, data in split_data.items():
    X = data["X"]
    y = data["y"]
    participant_ids = data["participant_object_id"]

    tensor_summary_rows.append(
        {
            "split": split_name,
            "path": str(data["path"].relative_to(PROJECT_ROOT)),
            "X_shape": X.shape,
            "samples": int(X.shape[0]),
            "features": int(X.shape[1]),
            "participants": int(pd.Series(participant_ids).nunique()),
            "target_0": int((y == 0).sum()),
            "target_1": int((y == 1).sum()),
            "target_mean": float(y.mean()),
            "nan_count": int(np.isnan(X).sum()),
            "inf_count": int(np.isinf(X).sum()),
        }
    )

tensor_summary_df = pd.DataFrame(tensor_summary_rows)

feature_check = pd.DataFrame(
    [
        {"metric": "metadata_final_feature_count", "value": metadata["zero_variance_removal"]["final_feature_count"]},
        {"metric": "feature_columns_csv_rows", "value": len(feature_columns_df)},
        {"metric": "npz_feature_count", "value": len(loaded_feature_columns)},
        {"metric": "train_X_feature_count", "value": split_data["train"]["X"].shape[1]},
    ]
)

problem_rows = tensor_summary_df[
    (tensor_summary_df["nan_count"] > 0)
    | (tensor_summary_df["inf_count"] > 0)
]

print("=== Pre-Sleep Stage 1 Training Setup ===")
print("DEVICE:", DEVICE)
print("RUN_PRE_SLEEP_STAGE1_TRAINING:", RUN_PRE_SLEEP_STAGE1_TRAINING)
print("EXPERIMENT_ID:", EXPERIMENT_ID)
print("TENSOR_DIR:", TENSOR_DIR)

print("\n=== Tensor Summary ===")
display(tensor_summary_df)

print("\n=== Feature Check ===")
display(feature_check)

print("\n=== Feature Group Counts ===")
display(feature_columns_df["feature_group"].value_counts().reset_index(name="features"))

print("\n=== Problem Rows ===")
if len(problem_rows) > 0:
    display(problem_rows)
else:
    print("없음")

print("\nReady for next step: define MLP model and training utilities.")

=== Pre-Sleep Stage 1 Training Setup ===
DEVICE: cpu
RUN_PRE_SLEEP_STAGE1_TRAINING: False
EXPERIMENT_ID: presleep_stage1_000
TENSOR_DIR: C:\workSpace\DeepLearnin_sleep\data\processed\pre_sleep_forecasting\design_c_stage1\mlp_current_day_final

=== Tensor Summary ===


,split,path,X_shape,samples,features,participants,target_0,target_1,target_mean,nan_count,inf_count
0,train,data\processed\pre_sleep_forecasting\design_c_...,"(2323, 58)",2323,58,46,1365,958,0.412398,0,0
1,validation,data\processed\pre_sleep_forecasting\design_c_...,"(347, 58)",347,58,9,225,122,0.351585,0,0
2,test,data\processed\pre_sleep_forecasting\design_c_...,"(881, 58)",881,58,14,563,318,0.360953,0,0



=== Feature Check ===


,metric,value
0,metadata_final_feature_count,58
1,feature_columns_csv_rows,58
2,npz_feature_count,58
3,train_X_feature_count,58



=== Feature Group Counts ===


,feature_group,features
0,missing_indicator,25
1,pre_sleep_intraday,18
2,previous_day_daily,9
3,timing,6



=== Problem Rows ===
없음

Ready for next step: define MLP model and training utilities.


In [2]:
# Cell 2. Define MLP model, training loop, and evaluation utilities
# 원하는 결과:
# - pre-sleep Stage 1 MLP 모델 구조를 정의한다.
# - validation balanced accuracy 기준 early stopping 학습 함수를 준비한다.
# - threshold search/evaluation 함수를 준비한다.
# - 아직 학습은 하지 않는다.

class PreSleepMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.25):
        super().__init__()

        layers = []
        current_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend(
                [
                    nn.Linear(current_dim, hidden_dim),
                    nn.BatchNorm1d(hidden_dim),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            )
            current_dim = hidden_dim

        layers.append(nn.Linear(current_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)

def make_loader(X, y, batch_size=64, shuffle=False):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def predict_proba(model, X, batch_size=256):
    model.eval()
    probabilities = []

    loader = make_loader(X, np.zeros(X.shape[0]), batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for batch_X, _ in loader:
            batch_X = batch_X.to(DEVICE)
            logits = model(batch_X)
            batch_probabilities = torch.sigmoid(logits).detach().cpu().numpy()
            probabilities.append(batch_probabilities)

    return np.concatenate(probabilities)

def evaluate_binary_predictions(y_true, y_probability, threshold):
    y_pred = (y_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    metrics = {
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    if len(np.unique(y_true)) == 2:
        metrics["roc_auc"] = roc_auc_score(y_true, y_probability)
        metrics["average_precision"] = average_precision_score(y_true, y_probability)
    else:
        metrics["roc_auc"] = np.nan
        metrics["average_precision"] = np.nan

    return metrics

def find_best_threshold(y_true, y_probability, metric_name="balanced_accuracy"):
    threshold_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

    rows = []
    for threshold in threshold_grid:
        row = evaluate_binary_predictions(y_true, y_probability, threshold)
        rows.append(row)

    threshold_df = pd.DataFrame(rows)

    best_row = (
        threshold_df
        .sort_values(
            [metric_name, "f1", "balanced_accuracy"],
            ascending=[False, False, False],
        )
        .iloc[0]
        .to_dict()
    )

    return float(best_row["threshold"]), threshold_df, best_row

def train_one_model(
    experiment_id,
    X_train,
    y_train,
    X_validation,
    y_validation,
    input_dim,
    seed=2026,
    max_epochs=100,
    patience=20,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    hidden_dims=(64, 32),
    dropout=0.25,
):
    set_seed(seed)

    model = PreSleepMLP(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
    ).to(DEVICE)

    positive_count = float((y_train == 1).sum())
    negative_count = float((y_train == 0).sum())
    pos_weight = torch.tensor([negative_count / max(positive_count, 1.0)], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    train_loader = make_loader(
        X_train,
        y_train,
        batch_size=batch_size,
        shuffle=True,
    )

    best_state = None
    best_validation_balanced_accuracy = -np.inf
    best_epoch = None
    best_threshold = None
    epochs_without_improvement = 0
    history_rows = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(DEVICE)
            batch_y = batch_y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(batch_X)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        validation_probability = predict_proba(model, X_validation)
        threshold, threshold_df, best_threshold_row = find_best_threshold(
            y_validation,
            validation_probability,
            metric_name="balanced_accuracy",
        )

        validation_metrics = evaluate_binary_predictions(
            y_validation,
            validation_probability,
            threshold,
        )

        mean_train_loss = float(np.mean(train_losses))

        history_row = {
            "experiment_id": experiment_id,
            "epoch": epoch,
            "train_loss": mean_train_loss,
            "validation_selected_threshold": threshold,
            **{
                f"validation_{key}": value
                for key, value in validation_metrics.items()
            },
        }
        history_rows.append(history_row)

        current_validation_balanced_accuracy = validation_metrics["balanced_accuracy"]

        if current_validation_balanced_accuracy > best_validation_balanced_accuracy:
            best_validation_balanced_accuracy = current_validation_balanced_accuracy
            best_epoch = epoch
            best_threshold = threshold
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0:
            print(
                f"{experiment_id} epoch {epoch:03d} | "
                f"loss={mean_train_loss:.4f} | "
                f"val_bal_acc={validation_metrics['balanced_accuracy']:.4f} | "
                f"val_auc={validation_metrics['roc_auc']:.4f} | "
                f"thr={threshold:.2f}"
            )

        if epochs_without_improvement >= patience:
            print(
                f"{experiment_id} early stopped at epoch {epoch}; "
                f"best_epoch={best_epoch}, "
                f"best_val_bal_acc={best_validation_balanced_accuracy:.4f}"
            )
            break

    model.load_state_dict(best_state)

    history_df = pd.DataFrame(history_rows)

    return {
        "model": model,
        "history_df": history_df,
        "best_epoch": best_epoch,
        "best_threshold": best_threshold,
        "best_validation_balanced_accuracy": best_validation_balanced_accuracy,
    }

print("Training utilities are ready.")
print("Model:", PreSleepMLP(input_dim=split_data["train"]["X"].shape[1]))
print("Next step: set RUN_PRE_SLEEP_STAGE1_TRAINING = True in Cell 3 to train manually.")

Training utilities are ready.
Model: PreSleepMLP(
  (network): Sequential(
    (0): Linear(in_features=58, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.25, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.25, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)
Next step: set RUN_PRE_SLEEP_STAGE1_TRAINING = True in Cell 3 to train manually.


In [4]:
# Cell 3. Train Pre-sleep Design C Stage 1 MLP
# 원하는 결과:
# - Stage 1 strict pre-sleep MLP를 학습한다.
# - validation balanced accuracy 기준 best epoch/threshold를 선택한다.
# - validation-selected threshold로 train/validation/test metrics를 저장한다.
# - prediction/history/model 파일을 저장한다.
# - log/YYYY-MM-DD.md를 업데이트한다.

RUN_PRE_SLEEP_STAGE1_TRAINING = True

MAX_EPOCHS = 100
PATIENCE = 20
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN_DIMS = (64, 32)
DROPOUT = 0.25

METRICS_PATH = OUTPUT_DIR / "pre_sleep_stage1_mlp_metrics.csv"
HISTORY_PATH = OUTPUT_DIR / "pre_sleep_stage1_mlp_training_history.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "pre_sleep_stage1_mlp_predictions.csv"
MODEL_PATH = MODEL_DIR / f"{EXPERIMENT_ID}_best.pt"
LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

if not RUN_PRE_SLEEP_STAGE1_TRAINING:
    print("RUN_PRE_SLEEP_STAGE1_TRAINING is False.")
    print("Set RUN_PRE_SLEEP_STAGE1_TRAINING = True to train manually.")
else:
    print("=== Pre-Sleep Stage 1 MLP Training Started ===")
    print("device:", DEVICE)
    print("experiment_id:", EXPERIMENT_ID)
    print("max_epochs:", MAX_EPOCHS)
    print("patience:", PATIENCE)
    print("batch_size:", BATCH_SIZE)
    print("learning_rate:", LEARNING_RATE)
    print("weight_decay:", WEIGHT_DECAY)
    print("hidden_dims:", HIDDEN_DIMS)
    print("dropout:", DROPOUT)

    train_result = train_one_model(
        experiment_id=EXPERIMENT_ID,
        X_train=split_data["train"]["X"],
        y_train=split_data["train"]["y"],
        X_validation=split_data["validation"]["X"],
        y_validation=split_data["validation"]["y"],
        input_dim=split_data["train"]["X"].shape[1],
        seed=SEED,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
    )

    model = train_result["model"]
    best_epoch = train_result["best_epoch"]
    selected_threshold = train_result["best_threshold"]
    history_df = train_result["history_df"]

    torch.save(
        {
            "experiment_id": EXPERIMENT_ID,
            "model_family": MODEL_FAMILY,
            "feature_timing": FEATURE_TIMING,
            "subset_name": SUBSET_NAME,
            "window": WINDOW,
            "seed": SEED,
            "input_dim": split_data["train"]["X"].shape[1],
            "hidden_dims": HIDDEN_DIMS,
            "dropout": DROPOUT,
            "best_epoch": best_epoch,
            "selected_threshold_from_validation": selected_threshold,
            "state_dict": model.state_dict(),
            "feature_columns": loaded_feature_columns,
        },
        MODEL_PATH,
    )

    metrics_rows = []
    prediction_rows = []

    for split_name in ["train", "validation", "test"]:
        X_split = split_data[split_name]["X"]
        y_split = split_data[split_name]["y"]
        probability = predict_proba(model, X_split)

        metrics = evaluate_binary_predictions(
            y_split,
            probability,
            selected_threshold,
        )

        metrics_rows.append(
            {
                "experiment_id": EXPERIMENT_ID,
                "feature_timing": FEATURE_TIMING,
                "subset_name": SUBSET_NAME,
                "model_family": MODEL_FAMILY,
                "window": WINDOW,
                "split": split_name,
                "seed": SEED,
                "best_epoch": best_epoch,
                "selected_threshold_from_validation": selected_threshold,
                **metrics,
            }
        )

        split_prediction_df = pd.DataFrame(
            {
                "experiment_id": EXPERIMENT_ID,
                "split": split_name,
                "sleep_episode_id": split_data[split_name]["sleep_episode_id"],
                "participant_object_id": split_data[split_name]["participant_object_id"],
                "y_true": y_split,
                "y_probability": probability,
                "selected_threshold_from_validation": selected_threshold,
                "y_pred": (probability >= selected_threshold).astype(int),
            }
        )
        prediction_rows.append(split_prediction_df)

    metrics_df = pd.DataFrame(metrics_rows)
    predictions_df = pd.concat(prediction_rows, ignore_index=True)

    metrics_df.to_csv(METRICS_PATH, index=False, encoding="utf-8-sig")
    history_df.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")
    predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

    metadata["stage1_training"] = {
        "experiment_id": EXPERIMENT_ID,
        "model_family": MODEL_FAMILY,
        "feature_timing": FEATURE_TIMING,
        "subset_name": SUBSET_NAME,
        "window": WINDOW,
        "seed": SEED,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "hidden_dims": list(HIDDEN_DIMS),
        "dropout": DROPOUT,
        "best_epoch": int(best_epoch),
        "selected_threshold_from_validation": float(selected_threshold),
        "metrics_path": str(METRICS_PATH.relative_to(PROJECT_ROOT)),
        "history_path": str(HISTORY_PATH.relative_to(PROJECT_ROOT)),
        "predictions_path": str(PREDICTIONS_PATH.relative_to(PROJECT_ROOT)),
        "model_path": str(MODEL_PATH.relative_to(PROJECT_ROOT)),
    }
    METADATA_PATH.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    test_metrics = metrics_df.loc[metrics_df["split"] == "test"].iloc[0]

    log_entry = f"""

## 2026-06-29 - Pre-sleep Design C Stage 1 MLP training

Trained first strict pre-sleep forecasting MLP.

- Notebook: `notebooks/12_pre_sleep_stage1_training.ipynb`
- Experiment: `{EXPERIMENT_ID}`
- Objective: predict `good_sleep_label` for sleep episode using features available before `sleep_start_datetime`
- Feature timing: `{FEATURE_TIMING}`
- Subset: `{SUBSET_NAME}`
- Model: `{MODEL_FAMILY}`
- Features: {split_data["train"]["X"].shape[1]}
- Train/validation/test samples: {split_data["train"]["X"].shape[0]} / {split_data["validation"]["X"].shape[0]} / {split_data["test"]["X"].shape[0]}
- Best epoch: {best_epoch}
- Selected threshold from validation: {selected_threshold:.2f}
- Test balanced accuracy: {test_metrics["balanced_accuracy"]:.4f}
- Test ROC AUC: {test_metrics["roc_auc"]:.4f}
- Test average precision: {test_metrics["average_precision"]:.4f}
- Test F1: {test_metrics["f1"]:.4f}
- Test precision: {test_metrics["precision"]:.4f}
- Test recall: {test_metrics["recall"]:.4f}
- Metrics: `{METRICS_PATH.relative_to(PROJECT_ROOT)}`
- History: `{HISTORY_PATH.relative_to(PROJECT_ROOT)}`
- Predictions: `{PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}`
- Model: `{MODEL_PATH.relative_to(PROJECT_ROOT)}`
"""

    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(log_entry)

    print("\n=== Pre-Sleep Stage 1 MLP Training Finished ===")
    print("model:", MODEL_PATH.relative_to(PROJECT_ROOT), MODEL_PATH.exists())
    print("metrics:", METRICS_PATH.relative_to(PROJECT_ROOT), METRICS_PATH.exists())
    print("history:", HISTORY_PATH.relative_to(PROJECT_ROOT), HISTORY_PATH.exists())
    print("predictions:", PREDICTIONS_PATH.relative_to(PROJECT_ROOT), PREDICTIONS_PATH.exists())

    print("\n=== Metrics ===")
    display(metrics_df)

    print("\n=== Validation Ranking Row ===")
    display(metrics_df.loc[metrics_df["split"] == "validation"])

    print("\n=== Test Row ===")
    display(metrics_df.loc[metrics_df["split"] == "test"])

    print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Pre-Sleep Stage 1 MLP Training Started ===
device: cpu
experiment_id: presleep_stage1_000
max_epochs: 100
patience: 20
batch_size: 64
learning_rate: 0.001
weight_decay: 0.0001
hidden_dims: (64, 32)
dropout: 0.25
presleep_stage1_000 epoch 001 | loss=0.7884 | val_bal_acc=0.6545 | val_auc=0.6739 | thr=0.47
presleep_stage1_000 epoch 010 | loss=0.7105 | val_bal_acc=0.6493 | val_auc=0.6736 | thr=0.48
presleep_stage1_000 epoch 020 | loss=0.6709 | val_bal_acc=0.6568 | val_auc=0.6745 | thr=0.44
presleep_stage1_000 early stopped at epoch 23; best_epoch=3, best_val_bal_acc=0.6718

=== Pre-Sleep Stage 1 MLP Training Finished ===
model: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_mlp_outputs\models\presleep_stage1_000_best.pt True
metrics: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_mlp_outputs\pre_sleep_stage1_mlp_metrics.csv True
history: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_mlp_outputs\pre_sleep_stage1

,experiment_id,feature_timing,subset_name,model_family,window,split,seed,best_epoch,selected_threshold_from_validation,threshold,balanced_accuracy,f1,precision,recall,tn,fp,fn,tp,roc_auc,average_precision
0,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,train,2026,3,0.55,0.55,0.670353,0.604550,0.628378,0.582463,1035,330,400,558,0.732512,0.652051
1,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,validation,2026,3,0.55,0.55,0.671821,0.581395,0.551471,0.614754,164,61,47,75,0.679891,0.468893
2,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,2026,3,0.55,0.55,0.633762,0.490421,0.627451,0.402516,487,76,190,128,0.687501,0.600856



=== Validation Ranking Row ===


,experiment_id,feature_timing,subset_name,model_family,window,split,seed,best_epoch,selected_threshold_from_validation,threshold,balanced_accuracy,f1,precision,recall,tn,fp,fn,tp,roc_auc,average_precision
1,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,validation,2026,3,0.55,0.55,0.671821,0.581395,0.551471,0.614754,164,61,47,75,0.679891,0.468893



=== Test Row ===


,experiment_id,feature_timing,subset_name,model_family,window,split,seed,best_epoch,selected_threshold_from_validation,threshold,balanced_accuracy,f1,precision,recall,tn,fp,fn,tp,roc_auc,average_precision
2,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,2026,3,0.55,0.55,0.633762,0.490421,0.627451,0.402516,487,76,190,128,0.687501,0.600856



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [5]:
# Cell 4. Threshold sensitivity for Pre-sleep Stage 1 MLP
# 원하는 결과:
# - 저장된 prediction을 사용해 threshold별 validation/test 성능을 계산한다.
# - validation balanced accuracy / validation F1 / validation recall-priority threshold 정책을 비교한다.
# - test-only best threshold는 diagnostic only로 분리한다.
# - threshold sensitivity 결과를 저장하고 log를 업데이트한다.

THRESHOLD_SENSITIVITY_PATH = OUTPUT_DIR / "pre_sleep_stage1_threshold_sensitivity.csv"
THRESHOLD_POLICY_COMPARISON_PATH = OUTPUT_DIR / "pre_sleep_stage1_threshold_policy_comparison.csv"

predictions_df = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")

threshold_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)

sensitivity_rows = []

for split_name in ["train", "validation", "test"]:
    split_predictions = predictions_df[predictions_df["split"] == split_name].copy()
    y_true = split_predictions["y_true"].to_numpy()
    y_probability = split_predictions["y_probability"].to_numpy()

    for threshold in threshold_grid:
        metrics = evaluate_binary_predictions(
            y_true=y_true,
            y_probability=y_probability,
            threshold=threshold,
        )
        sensitivity_rows.append(
            {
                "experiment_id": EXPERIMENT_ID,
                "split": split_name,
                **metrics,
            }
        )

threshold_sensitivity_df = pd.DataFrame(sensitivity_rows)

validation_sensitivity_df = threshold_sensitivity_df[
    threshold_sensitivity_df["split"] == "validation"
].copy()

test_sensitivity_df = threshold_sensitivity_df[
    threshold_sensitivity_df["split"] == "test"
].copy()

def select_threshold_policy(validation_df, policy_name):
    if policy_name == "validation_balanced_accuracy":
        return (
            validation_df
            .sort_values(["balanced_accuracy", "f1", "precision"], ascending=[False, False, False])
            .iloc[0]
        )

    if policy_name == "validation_f1":
        return (
            validation_df
            .sort_values(["f1", "balanced_accuracy", "precision"], ascending=[False, False, False])
            .iloc[0]
        )

    if policy_name == "validation_recall_priority_precision_at_least_0_45":
        eligible = validation_df[validation_df["precision"] >= 0.45].copy()
        if len(eligible) == 0:
            eligible = validation_df.copy()
        return (
            eligible
            .sort_values(["recall", "balanced_accuracy", "f1"], ascending=[False, False, False])
            .iloc[0]
        )

    if policy_name == "validation_precision_priority_recall_at_least_0_40":
        eligible = validation_df[validation_df["recall"] >= 0.40].copy()
        if len(eligible) == 0:
            eligible = validation_df.copy()
        return (
            eligible
            .sort_values(["precision", "balanced_accuracy", "f1"], ascending=[False, False, False])
            .iloc[0]
        )

    raise ValueError(f"Unsupported policy: {policy_name}")

threshold_policies = [
    "validation_balanced_accuracy",
    "validation_f1",
    "validation_recall_priority_precision_at_least_0_45",
    "validation_precision_priority_recall_at_least_0_40",
]

policy_rows = []

for policy_name in threshold_policies:
    selected_validation_row = select_threshold_policy(
        validation_sensitivity_df,
        policy_name,
    )
    selected_threshold = float(selected_validation_row["threshold"])

    for split_name in ["validation", "test"]:
        split_row = threshold_sensitivity_df[
            (threshold_sensitivity_df["split"] == split_name)
            & (threshold_sensitivity_df["threshold"] == selected_threshold)
        ].iloc[0]

        policy_rows.append(
            {
                "experiment_id": EXPERIMENT_ID,
                "threshold_policy": policy_name,
                "selected_threshold": selected_threshold,
                "evaluation_split": split_name,
                "balanced_accuracy": split_row["balanced_accuracy"],
                "roc_auc": split_row["roc_auc"],
                "average_precision": split_row["average_precision"],
                "f1": split_row["f1"],
                "precision": split_row["precision"],
                "recall": split_row["recall"],
                "tn": int(split_row["tn"]),
                "fp": int(split_row["fp"]),
                "fn": int(split_row["fn"]),
                "tp": int(split_row["tp"]),
            }
        )

best_test_diagnostic_rows = []

for metric_name in ["balanced_accuracy", "f1", "precision", "recall"]:
    if metric_name == "recall":
        diagnostic_row = (
            test_sensitivity_df[test_sensitivity_df["precision"] >= 0.30]
            .sort_values(["recall", "balanced_accuracy"], ascending=[False, False])
            .iloc[0]
        )
        diagnostic_label = "test_recall_with_precision_at_least_0_30"
    else:
        diagnostic_row = (
            test_sensitivity_df
            .sort_values([metric_name, "balanced_accuracy"], ascending=[False, False])
            .iloc[0]
        )
        diagnostic_label = f"test_best_{metric_name}"

    best_test_diagnostic_rows.append(
        {
            "experiment_id": EXPERIMENT_ID,
            "diagnostic_policy": diagnostic_label,
            "threshold": diagnostic_row["threshold"],
            "balanced_accuracy": diagnostic_row["balanced_accuracy"],
            "roc_auc": diagnostic_row["roc_auc"],
            "average_precision": diagnostic_row["average_precision"],
            "f1": diagnostic_row["f1"],
            "precision": diagnostic_row["precision"],
            "recall": diagnostic_row["recall"],
            "tn": int(diagnostic_row["tn"]),
            "fp": int(diagnostic_row["fp"]),
            "fn": int(diagnostic_row["fn"]),
            "tp": int(diagnostic_row["tp"]),
        }
    )

threshold_policy_comparison_df = pd.DataFrame(policy_rows)
best_test_diagnostic_df = pd.DataFrame(best_test_diagnostic_rows)

threshold_sensitivity_df.to_csv(
    THRESHOLD_SENSITIVITY_PATH,
    index=False,
    encoding="utf-8-sig",
)
threshold_policy_comparison_df.to_csv(
    THRESHOLD_POLICY_COMPARISON_PATH,
    index=False,
    encoding="utf-8-sig",
)

metadata["stage1_threshold_sensitivity"] = {
    "threshold_sensitivity_path": str(THRESHOLD_SENSITIVITY_PATH.relative_to(PROJECT_ROOT)),
    "threshold_policy_comparison_path": str(THRESHOLD_POLICY_COMPARISON_PATH.relative_to(PROJECT_ROOT)),
    "threshold_grid": [float(x) for x in threshold_grid],
    "note": "Threshold policies are selected on validation only. Test-best rows are diagnostic only.",
}
METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

log_entry = f"""

## 2026-06-29 - Pre-sleep Stage 1 threshold sensitivity

Evaluated threshold sensitivity for `{EXPERIMENT_ID}`.

- Threshold sensitivity: `{THRESHOLD_SENSITIVITY_PATH.relative_to(PROJECT_ROOT)}`
- Threshold policy comparison: `{THRESHOLD_POLICY_COMPARISON_PATH.relative_to(PROJECT_ROOT)}`
- Policies: {", ".join(threshold_policies)}
- Test-only best threshold rows are diagnostic only and must not be used for model selection.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Threshold Sensitivity Saved ===")
print("threshold sensitivity:", THRESHOLD_SENSITIVITY_PATH.relative_to(PROJECT_ROOT), THRESHOLD_SENSITIVITY_PATH.exists())
print("policy comparison:", THRESHOLD_POLICY_COMPARISON_PATH.relative_to(PROJECT_ROOT), THRESHOLD_POLICY_COMPARISON_PATH.exists())

print("\n=== Threshold Policy Comparison: Validation Rows ===")
display(
    threshold_policy_comparison_df[
        threshold_policy_comparison_df["evaluation_split"] == "validation"
    ]
)

print("\n=== Threshold Policy Comparison: Test Rows ===")
display(
    threshold_policy_comparison_df[
        threshold_policy_comparison_df["evaluation_split"] == "test"
    ]
)

print("\n=== Best Test Thresholds Diagnostic Only ===")
display(best_test_diagnostic_df)

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Threshold Sensitivity Saved ===
threshold sensitivity: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_mlp_outputs\pre_sleep_stage1_threshold_sensitivity.csv True
policy comparison: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_mlp_outputs\pre_sleep_stage1_threshold_policy_comparison.csv True

=== Threshold Policy Comparison: Validation Rows ===


,experiment_id,threshold_policy,selected_threshold,evaluation_split,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,tn,fp,fn,tp
0,presleep_stage1_000,validation_balanced_accuracy,0.55,validation,0.671821,0.679891,0.468893,0.581395,0.551471,0.614754,164,61,47,75
2,presleep_stage1_000,validation_f1,0.35,validation,0.637832,0.679891,0.468893,0.589189,0.439516,0.893443,86,139,13,109
4,presleep_stage1_000,validation_recall_priority_precision_at_least_...,0.38,validation,0.637541,0.679891,0.468893,0.577381,0.453271,0.795082,108,117,25,97
6,presleep_stage1_000,validation_precision_priority_recall_at_least_...,0.59,validation,0.654590,0.679891,0.468893,0.551440,0.553719,0.549180,171,54,55,67



=== Threshold Policy Comparison: Test Rows ===


,experiment_id,threshold_policy,selected_threshold,evaluation_split,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,tn,fp,fn,tp
1,presleep_stage1_000,validation_balanced_accuracy,0.55,test,0.633762,0.687501,0.600856,0.490421,0.627451,0.402516,487,76,190,128
3,presleep_stage1_000,validation_f1,0.35,test,0.568604,0.687501,0.600856,0.545639,0.402695,0.845912,164,399,49,269
5,presleep_stage1_000,validation_recall_priority_precision_at_least_...,0.38,test,0.594069,0.687501,0.600856,0.554217,0.425210,0.795597,221,342,65,253
7,presleep_stage1_000,validation_precision_priority_recall_at_least_...,0.59,test,0.612144,0.687501,0.600856,0.424893,0.668919,0.311321,514,49,219,99



=== Best Test Thresholds Diagnostic Only ===


,experiment_id,diagnostic_policy,threshold,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,tn,fp,fn,tp
0,presleep_stage1_000,test_best_balanced_accuracy,0.49,0.657244,0.687501,0.600856,0.558587,0.570492,0.547170,432,131,144,174
1,presleep_stage1_000,test_best_f1,0.41,0.625750,0.687501,0.600856,0.569028,0.460194,0.745283,285,278,81,237
2,presleep_stage1_000,test_best_precision,0.78,0.517296,0.687501,0.600856,0.066869,1.000000,0.034591,563,0,307,11
3,presleep_stage1_000,test_recall_with_precision_at_least_0_30,0.05,0.500000,0.687501,0.600856,0.530442,0.360953,1.000000,0,563,0,318



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [7]:
# Cell 5. Seed robustness for Pre-sleep Design C Stage 1 MLP
# 원하는 결과:
# - 같은 MLP 구조를 여러 seed로 반복 학습한다.
# - validation-selected threshold 기준 test 성능의 평균/표준편차를 확인한다.
# - seed별 metrics/history/predictions/model을 저장한다.
# - log/YYYY-MM-DD.md를 업데이트한다.

RUN_PRE_SLEEP_STAGE1_SEED_ROBUSTNESS = True

ROBUSTNESS_SEEDS = [7, 123, 2027]
ROBUSTNESS_EXPERIMENT_PREFIX = "presleep_stage1_seed"

SEED_OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_mlp_seed_robustness_outputs"
SEED_MODEL_DIR = SEED_OUTPUT_DIR / "models"
SEED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED_METRICS_PATH = SEED_OUTPUT_DIR / "pre_sleep_stage1_seed_robustness_metrics.csv"
SEED_HISTORY_PATH = SEED_OUTPUT_DIR / "pre_sleep_stage1_seed_robustness_history.csv"
SEED_PREDICTIONS_PATH = SEED_OUTPUT_DIR / "pre_sleep_stage1_seed_robustness_predictions.csv"
SEED_SUMMARY_PATH = SEED_OUTPUT_DIR / "pre_sleep_stage1_seed_robustness_summary.csv"

if not RUN_PRE_SLEEP_STAGE1_SEED_ROBUSTNESS:
    print("RUN_PRE_SLEEP_STAGE1_SEED_ROBUSTNESS is False.")
    print("Set RUN_PRE_SLEEP_STAGE1_SEED_ROBUSTNESS = True to run seed robustness manually.")
else:
    all_seed_metrics = []
    all_seed_history = []
    all_seed_predictions = []

    print("=== Pre-Sleep Stage 1 Seed Robustness Started ===")
    print("seeds:", ROBUSTNESS_SEEDS)
    print("device:", DEVICE)

    for seed in ROBUSTNESS_SEEDS:
        seed_experiment_id = f"{ROBUSTNESS_EXPERIMENT_PREFIX}{seed}"
        seed_model_path = SEED_MODEL_DIR / f"{seed_experiment_id}_best.pt"

        print("\n" + "=" * 80)
        print(f"Training {seed_experiment_id}")
        print("=" * 80)

        train_result = train_one_model(
            experiment_id=seed_experiment_id,
            X_train=split_data["train"]["X"],
            y_train=split_data["train"]["y"],
            X_validation=split_data["validation"]["X"],
            y_validation=split_data["validation"]["y"],
            input_dim=split_data["train"]["X"].shape[1],
            seed=seed,
            max_epochs=MAX_EPOCHS,
            patience=PATIENCE,
            batch_size=BATCH_SIZE,
            learning_rate=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
            hidden_dims=HIDDEN_DIMS,
            dropout=DROPOUT,
        )

        model = train_result["model"]
        best_epoch = train_result["best_epoch"]
        selected_threshold = train_result["best_threshold"]
        history_df = train_result["history_df"].copy()
        history_df["seed"] = seed

        torch.save(
            {
                "experiment_id": seed_experiment_id,
                "base_experiment_id": EXPERIMENT_ID,
                "model_family": MODEL_FAMILY,
                "feature_timing": FEATURE_TIMING,
                "subset_name": SUBSET_NAME,
                "window": WINDOW,
                "seed": seed,
                "input_dim": split_data["train"]["X"].shape[1],
                "hidden_dims": HIDDEN_DIMS,
                "dropout": DROPOUT,
                "best_epoch": best_epoch,
                "selected_threshold_from_validation": selected_threshold,
                "state_dict": model.state_dict(),
                "feature_columns": loaded_feature_columns,
            },
            seed_model_path,
        )

        seed_metrics_rows = []
        seed_prediction_rows = []

        for split_name in ["train", "validation", "test"]:
            X_split = split_data[split_name]["X"]
            y_split = split_data[split_name]["y"]
            probability = predict_proba(model, X_split)

            metrics = evaluate_binary_predictions(
                y_split,
                probability,
                selected_threshold,
            )

            seed_metrics_rows.append(
                {
                    "experiment_id": seed_experiment_id,
                    "base_experiment_id": EXPERIMENT_ID,
                    "feature_timing": FEATURE_TIMING,
                    "subset_name": SUBSET_NAME,
                    "model_family": MODEL_FAMILY,
                    "window": WINDOW,
                    "split": split_name,
                    "seed": seed,
                    "best_epoch": best_epoch,
                    "selected_threshold_from_validation": selected_threshold,
                    **metrics,
                    "model_path": str(seed_model_path.relative_to(PROJECT_ROOT)),
                }
            )

            split_prediction_df = pd.DataFrame(
                {
                    "experiment_id": seed_experiment_id,
                    "base_experiment_id": EXPERIMENT_ID,
                    "split": split_name,
                    "seed": seed,
                    "sleep_episode_id": split_data[split_name]["sleep_episode_id"],
                    "participant_object_id": split_data[split_name]["participant_object_id"],
                    "y_true": y_split,
                    "y_probability": probability,
                    "selected_threshold_from_validation": selected_threshold,
                    "y_pred": (probability >= selected_threshold).astype(int),
                }
            )
            seed_prediction_rows.append(split_prediction_df)

        all_seed_metrics.append(pd.DataFrame(seed_metrics_rows))
        all_seed_history.append(history_df)
        all_seed_predictions.append(pd.concat(seed_prediction_rows, ignore_index=True))

    seed_metrics_df = pd.concat(all_seed_metrics, ignore_index=True)
    seed_history_df = pd.concat(all_seed_history, ignore_index=True)
    seed_predictions_df = pd.concat(all_seed_predictions, ignore_index=True)

    seed_test_metrics_df = seed_metrics_df[
        seed_metrics_df["split"] == "test"
    ].copy()

    seed_summary_df = (
        seed_test_metrics_df
        .groupby(["base_experiment_id", "feature_timing", "subset_name", "model_family", "window"])
        .agg(
            runs=("experiment_id", "nunique"),
            mean_test_balanced_accuracy=("balanced_accuracy", "mean"),
            std_test_balanced_accuracy=("balanced_accuracy", "std"),
            min_test_balanced_accuracy=("balanced_accuracy", "min"),
            max_test_balanced_accuracy=("balanced_accuracy", "max"),
            mean_test_roc_auc=("roc_auc", "mean"),
            std_test_roc_auc=("roc_auc", "std"),
            mean_test_average_precision=("average_precision", "mean"),
            mean_test_f1=("f1", "mean"),
            mean_test_precision=("precision", "mean"),
            mean_test_recall=("recall", "mean"),
            mean_selected_threshold=("selected_threshold_from_validation", "mean"),
            mean_best_epoch=("best_epoch", "mean"),
        )
        .reset_index()
    )

    reference_previous_day_balanced_accuracy = 0.609761
    current_single_seed_test_balanced_accuracy = float(
        metrics_df.loc[metrics_df["split"] == "test", "balanced_accuracy"].iloc[0]
    )

    seed_summary_df["reference_previous_day_balanced_accuracy"] = reference_previous_day_balanced_accuracy
    seed_summary_df["single_seed_test_balanced_accuracy"] = current_single_seed_test_balanced_accuracy
    seed_summary_df["delta_mean_vs_previous_day_reference"] = (
        seed_summary_df["mean_test_balanced_accuracy"]
        - reference_previous_day_balanced_accuracy
    )
    seed_summary_df["delta_mean_vs_single_seed"] = (
        seed_summary_df["mean_test_balanced_accuracy"]
        - current_single_seed_test_balanced_accuracy
    )

    seed_metrics_df.to_csv(SEED_METRICS_PATH, index=False, encoding="utf-8-sig")
    seed_history_df.to_csv(SEED_HISTORY_PATH, index=False, encoding="utf-8-sig")
    seed_predictions_df.to_csv(SEED_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")
    seed_summary_df.to_csv(SEED_SUMMARY_PATH, index=False, encoding="utf-8-sig")

    metadata["stage1_seed_robustness"] = {
        "seeds": ROBUSTNESS_SEEDS,
        "metrics_path": str(SEED_METRICS_PATH.relative_to(PROJECT_ROOT)),
        "history_path": str(SEED_HISTORY_PATH.relative_to(PROJECT_ROOT)),
        "predictions_path": str(SEED_PREDICTIONS_PATH.relative_to(PROJECT_ROOT)),
        "summary_path": str(SEED_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
        "reference_previous_day_balanced_accuracy": reference_previous_day_balanced_accuracy,
    }
    METADATA_PATH.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    log_entry = f"""

## 2026-06-29 - Pre-sleep Stage 1 MLP seed robustness

Ran seed robustness for strict pre-sleep Stage 1 MLP.

- Base experiment: `{EXPERIMENT_ID}`
- Seeds: {ROBUSTNESS_SEEDS}
- Metrics: `{SEED_METRICS_PATH.relative_to(PROJECT_ROOT)}`
- History: `{SEED_HISTORY_PATH.relative_to(PROJECT_ROOT)}`
- Predictions: `{SEED_PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}`
- Summary: `{SEED_SUMMARY_PATH.relative_to(PROJECT_ROOT)}`
- Mean test balanced accuracy: {seed_summary_df["mean_test_balanced_accuracy"].iloc[0]:.4f}
- Std test balanced accuracy: {seed_summary_df["std_test_balanced_accuracy"].iloc[0]:.4f}
- Delta mean vs previous-day reference: {seed_summary_df["delta_mean_vs_previous_day_reference"].iloc[0]:.4f}
"""

    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(log_entry)

    print("\n=== Pre-Sleep Stage 1 Seed Robustness Saved ===")
    print("metrics:", SEED_METRICS_PATH.relative_to(PROJECT_ROOT), SEED_METRICS_PATH.exists())
    print("history:", SEED_HISTORY_PATH.relative_to(PROJECT_ROOT), SEED_HISTORY_PATH.exists())
    print("predictions:", SEED_PREDICTIONS_PATH.relative_to(PROJECT_ROOT), SEED_PREDICTIONS_PATH.exists())
    print("summary:", SEED_SUMMARY_PATH.relative_to(PROJECT_ROOT), SEED_SUMMARY_PATH.exists())

    print("\n=== Seed Test Metrics ===")
    display(seed_test_metrics_df)

    print("\n=== Seed Robustness Summary ===")
    display(seed_summary_df)

    print("\nReference previous-day balanced accuracy:", reference_previous_day_balanced_accuracy)
    print("Single seed pre-sleep Stage 1 test balanced accuracy:", current_single_seed_test_balanced_accuracy)
    print("log updated:", LOG_PATH, LOG_PATH.exists())

=== Pre-Sleep Stage 1 Seed Robustness Started ===
seeds: [7, 123, 2027]
device: cpu

Training presleep_stage1_seed7
presleep_stage1_seed7 epoch 001 | loss=0.7902 | val_bal_acc=0.6780 | val_auc=0.6972 | thr=0.41
presleep_stage1_seed7 epoch 010 | loss=0.7016 | val_bal_acc=0.6539 | val_auc=0.6912 | thr=0.55
presleep_stage1_seed7 epoch 020 | loss=0.6772 | val_bal_acc=0.6615 | val_auc=0.6760 | thr=0.49
presleep_stage1_seed7 early stopped at epoch 21; best_epoch=1, best_val_bal_acc=0.6780

Training presleep_stage1_seed123
presleep_stage1_seed123 epoch 001 | loss=0.7977 | val_bal_acc=0.6711 | val_auc=0.7035 | thr=0.51
presleep_stage1_seed123 epoch 010 | loss=0.7035 | val_bal_acc=0.6536 | val_auc=0.6841 | thr=0.52
presleep_stage1_seed123 epoch 020 | loss=0.6767 | val_bal_acc=0.6430 | val_auc=0.6632 | thr=0.60
presleep_stage1_seed123 early stopped at epoch 21; best_epoch=1, best_val_bal_acc=0.6711

Training presleep_stage1_seed2027
presleep_stage1_seed2027 epoch 001 | loss=0.8085 | val_bal_acc=

,experiment_id,base_experiment_id,feature_timing,subset_name,model_family,window,split,seed,best_epoch,selected_threshold_from_validation,...,f1,precision,recall,tn,fp,fn,tp,roc_auc,average_precision,model_path
2,presleep_stage1_seed7,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,7,1,0.41,...,0.530850,0.421442,0.716981,250,313,90,228,0.655965,0.590770,data\processed\pre_sleep_forecasting\design_c_...
5,presleep_stage1_seed123,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,123,1,0.51,...,0.510949,0.608696,0.440252,473,90,178,140,0.664784,0.605468,data\processed\pre_sleep_forecasting\design_c_...
8,presleep_stage1_seed2027,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,test,2027,1,0.43,...,0.551046,0.452525,0.704403,292,271,94,224,0.683652,0.608633,data\processed\pre_sleep_forecasting\design_c_...



=== Seed Robustness Summary ===


,base_experiment_id,feature_timing,subset_name,model_family,window,runs,mean_test_balanced_accuracy,std_test_balanced_accuracy,min_test_balanced_accuracy,max_test_balanced_accuracy,...,mean_test_average_precision,mean_test_f1,mean_test_precision,mean_test_recall,mean_selected_threshold,mean_best_epoch,reference_previous_day_balanced_accuracy,single_seed_test_balanced_accuracy,delta_mean_vs_previous_day_reference,delta_mean_vs_single_seed
0,presleep_stage1_000,pre_sleep,design_c_stage1_intraday_previous_day,mlp_current_day,1,3,0.610746,0.029848,0.580515,0.640197,...,0.601623,0.530948,0.494221,0.620545,0.45,1.0,0.609761,0.633762,0.000985,-0.023016



Reference previous-day balanced accuracy: 0.609761
Single seed pre-sleep Stage 1 test balanced accuracy: 0.6337623021325558
log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
